# CFRM 422 Strategy Project Notebook
## Multi-Asset News Sentiment Trading with Modern NLP

This study develops, tests, and critiques a multi-asset trading strategy that uses modern natural-language-processing models to extract asset-day sentiment signals from financial news. The empirical design is intentionally close to the economic mechanism in Tetlock-style media-pessimism studies, but extends that framework in three directions: a broader cross-section of equities, modern transformer-based sentiment models, and a walk-forward forecasting/backtesting pipeline that separates **indicators**, **signal formation**, and **trading rules**.

The goal is to determine whether news tone contains economically meaningful information while considering for implementation frictions and overfitting risk.


## 1. Summary of hypotheses and tests


### 1.1 Overall strategy theory

The strategy is a modern adaptation of the Tetlock insight that news tone can predict short-horizon market behavior through temporary sentiment pressure, delayed reaction, or attention effects. The innovation is to replace older lexicon-style language measurement with modern sentiment models, while preserving a disciplined implementation structure: asset-linked news is converted into daily sentiment indicators, those indicators are translated into a probabilistic forecast, and the forecast is mapped into simple trading rules. Tetlock's paper itself emphasizes that a basic sentiment-based trading rule can be informative as a benchmark, but also warns that transaction costs and implementation frictions may eliminate economic value.

### 1.2 Primary research focus

Does a finance-specific sentiment model applied to asset-linked daily news improve short-horizon forecasting and trading performance relative to:
1. a market-only model with no text,
2. a generic sentiment transformer,
3. a finance lexicon baseline, and
4. a simple Tetlock-style indicator rule?

### 1.3 Hypotheses and primary tests

To keep the notebook streamlined, each non-central hypothesis is evaluated with at most two primary tests. Additional tables and plots are included as diagnostics.

#### H1. Indicator hypothesis
Finance-aware sentiment indicators should display stronger economically sensible relations than generic or lexicon-based indicators.

**Primary tests**
1. **Return-screening test:** compare pooled indicator-return relation using simple association and bucket monotonicity.
2. **Activity-channel test:** test whether absolute sentiment predicts next-day trading activity, consistent with an attention / sentiment mechanism.

#### H2. Signal-process hypothesis
Sentiment should add incremental predictive information **after** conditioning on lagged market controls.

**Tests**
1. **Forecast-quality test:** compare `baseline_market_only` against `sentiment_plus_controls` out of sample using probability/discrimination metrics.
2. **Ranking-value test:** test whether higher forecast scores correspond to higher subsequent returns via a top-minus-bottom score spread.

#### H3. Rule-process hypothesis
If the signal is economically meaningful, it should remain useful when translated into a simple, implementable rule.

**Tests**
1. **Primary rule test:** evaluate the cross-sectional long-short ranking rule after transaction costs.
2. **Benchmark rule test:** compare the primary rule with a simpler Tetlock-style rolling-tercile rule.

#### H4. Optimization hypothesis
A sensible specification should not depend on a hyper-specific parameter choice.

**Tests**
1. **Validation-slice tuning test:** assess a small, theory-motivated grid over `C` and `top_n`.
2. **Local-stability test:** check whether nearby parameter choices result in similar performance.

#### H5. Walk-forward / objective-function hypothesis
A valid strategy should be selected with time-respecting procedures and evaluated using objectives appropriate to each layer of the problem.

**Tests**
1. **Walk-forward predictive test:** repeated rolling-origin predictive evaluation.
2. **Objective-alignment test:** compare whether parameter choices favored by predictive metrics also hold up under net economic metrics.

#### H6. Overfitting hypothesis
If the result is valuable, it should not rely on one regime, one cost assumption, or one fragile spec.

**Tests**
1. **Time-stability test:** yearly/fold-level stability of performance.
2. **Fragility test:** cost and parameter sensitivity of the preferred specification.

#### H7. Extension hypothesis
If the central result reflects a genuine mechanism, it should hold up under two Tetlock-based extensions.

**Tests**
1. **Reversal-horizon extension:** evaluate whether pessimism is associated with pressure and subsequent reversal across short horizons.
2. **Model-comparison extension:** compare finance-specific, generic, and lexicon-based sentiment sources under the same forecasting and rule framework.

### 1.4 Pre-specified primary specification

For simplicity, the following specification will be considered the primary design; all else can be considered a benchmark, diagnostic, or extension:

- **sentiment source:** FinBERT  
- **feature set:** sentiment-plus-controls  
- **signal model:** logistic regression  
- **target:** next-day open-to-close return sign  
- **rule:** cross-sectional long top-`N` / short bottom-`N`  
- **primary cost assumption:** 5 bps round-trip  
- **evaluation mode:** walk-forward out of sample

This is meant to reduce specification drift and to keep the final conclusion tied to a single clearly stated strategy rather than to the best one after testing.

## 2. Constraints, benchmarks, and objectives

### 2.1 Objectives

This project has two distinct but related objectives.

1. **Statistical objective.** Determine whether modern sentiment indicators improve next-day return forecasting beyond lagged return and activity controls.
2. **Economic objective.** Determine whether any forecasting improvement survives translation into a transparent, cost-aware long-short trading rule.

### 2.2 Constraints

The design is intentionally constrained to remain implementable and auditable:

- **Information timing.** Day-\(t\) features may use only information available by the close of day \(t\); returns on day \(t+1\) are the forecast target.
- **Universe quality.** Only tickers with sufficient news coverage and usable price data are retained.
- **Simple core model.** Logistic regression is the primary signal model; more flexible learners are extensions.
- **Simple core rule.** The main trading rule is equal-weighted and cross-sectional, avoiding hidden optimizer complexity.
- **Explicit frictions.** Economic conclusions are based on cost-adjusted, not just gross, returns.
- **Runtime realism.** The project favors a disciplined subset of models and tests over exhaustive search.

### 2.3 Benchmarks

#### Forecast benchmarks
- **Market-only baseline:** asks whether lagged return and activity controls already capture meaningful signal.
- **Sentiment-only baseline:** asks whether text alone has standalone predictive value.
- **Alternative text benchmarks:** generic-transformer sentiment and Loughran–McDonald lexicon sentiment test whether domain-specific NLP is actually incremental.

#### Trading benchmarks
- **Tetlock-style rolling-tercile rule:** a low-complexity news-tone benchmark designed to remain close to the original paper's spirit.
- **Cross-model comparison under the same rule:** asks whether any trading value comes from better text extraction rather than rule engineering.


## Environment setup and package installation

Only run this if the environment is missing any required libraries.


In [ ]:
!pip install pandas numpy matplotlib scikit-learn statsmodels yfinance transformers torch xgboost

In [ ]:
import os
import re
import math
import json
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple, Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    confusion_matrix,
    log_loss,
    brier_score_loss
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from scipy import stats
import statsmodels.api as sm

warnings.filterwarnings("ignore")

try:
    import yfinance as yf
except Exception:
    yf = None

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
except Exception:
    torch = None
    AutoTokenizer = None
    AutoModelForSequenceClassification = None
    AutoConfig = None

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    XGBClassifier = None

plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

if torch is not None:
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
else:
    DEVICE = "cpu"

print(f"DEVICE = {DEVICE}")
print(f"HAS_XGB = {HAS_XGB}")

In [ ]:
CONFIG = {
    # Dataset File Paths
    "news_data_path": "./analyst_ratings_processed.csv",   # asset news dataset
    "lm_negative_words_path": "./Loughran-McDonald_MasterDictionary_1993-2024.csv", # LM dictionary csv/txt

    # Data specification
    "start_date": "2009-01-01",
    "end_date": "2020-12-31",
    "max_tickers": 40, # keeps the project computationally manageable
    "min_news_per_ticker": 40,
    "headline_only": True, # headline/title text if body not available

    # Sentiment scoring toggles
    "run_finbert": True,
    "run_siebert": True,
    "run_lexicon": True,

    # Model / backtest settings
    "target_return_type": "next_open_to_close",
    "trading_days_per_year": 252,
    "train_min_days": 252 * 2,
    "validation_days": 63,
    "test_days": 63,
    "top_n": 5,
    "cost_bps_list": [0, 5, 10, 25], # round-trip cost sensitivity
    "prob_threshold_long": 0.55,
    "prob_threshold_short": 0.45,

    # Inference settings
    "transformer_batch_size": 32,
    "transformer_max_length": 96,

    # Export
    "results_dir": "strategy_project_outputs",
}

os.makedirs(CONFIG["results_dir"], exist_ok=True)
CONFIG

## Reproducibility utilities

These helper functions standardize file loading, exporting, plotting support, and statistical calculations used throughout the notebook.

In [ ]:
def first_existing_path(paths: List[str]) -> Optional[str]:
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def infer_column(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    lowered = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lowered:
            return lowered[cand.lower()]
    if required:
        raise KeyError(f"Could not find any of {candidates} in columns: {list(df.columns)}")
    return None

def clean_text(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"http\S+|www\S+", " ", x)
    x = re.sub(r"<.*?>", " ", x)
    x = re.sub(r"[^\w\s\$%\-\+\.]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def normalize_ticker(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).upper().strip()
    x = x.replace("$", "")
    x = re.sub(r"[^A-Z\.\-]", "", x)
    return x if x else np.nan

def safe_log_return(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return np.log(a / b)

def compute_max_drawdown(returns: pd.Series) -> float:
    equity = (1 + returns.fillna(0)).cumprod()
    peak = equity.cummax()
    dd = equity / peak - 1
    return float(dd.min()) if len(dd) else np.nan

def annualized_sharpe(returns: pd.Series, periods_per_year: int = 252) -> float:
    r = returns.dropna()
    if len(r) < 2 or r.std() == 0:
        return np.nan
    return float(np.sqrt(periods_per_year) * r.mean() / r.std())

def annualized_return(returns: pd.Series, periods_per_year: int = 252) -> float:
    r = returns.dropna()
    if len(r) == 0:
        return np.nan
    growth = (1 + r).prod()
    years = len(r) / periods_per_year
    if years <= 0 or growth <= 0:
        return np.nan
    return float(growth ** (1 / years) - 1)

def annualized_vol(returns: pd.Series, periods_per_year: int = 252) -> float:
    r = returns.dropna()
    if len(r) < 2:
        return np.nan
    return float(r.std() * np.sqrt(periods_per_year))

def pesaran_timmermann_test(y_true: pd.Series, y_pred_binary: pd.Series) -> Dict[str, float]:
    '''
    Directional accuracy test
    '''
    y = pd.Series(y_true).astype(int).reset_index(drop=True)
    z = pd.Series(y_pred_binary).astype(int).reset_index(drop=True)
    mask = y.notna() & z.notna()
    y = y[mask]
    z = z[mask]
    n = len(y)
    if n < 10:
        return {"pt_stat": np.nan, "p_value": np.nan, "hit_rate": np.nan}

    py = y.mean()
    pz = z.mean()
    pc = (y == z).mean()
    p = py * pz + (1 - py) * (1 - pz)
    v = p * (1 - p) / n
    qy = py * (1 - py) / n
    qz = pz * (1 - pz) / n
    w = ((2 * py - 1) ** 2) * qz + ((2 * pz - 1) ** 2) * qy + 4 * qy * qz

    denom = max(v - w, 1e-12)
    pt = (pc - p) / np.sqrt(denom)
    pval = 2 * (1 - stats.norm.cdf(abs(pt)))
    return {"pt_stat": float(pt), "p_value": float(pval), "hit_rate": float(pc)}

def save_csv(df: pd.DataFrame, filename: str):
    path = os.path.join(CONFIG["results_dir"], filename)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

## 3. Data Setup & Description

### 3.1 Data inputs

#### 1. Asset-linked news
The central text input is a large panel of financial news headlines with publication dates and ticker mappings from 2009-2020 for thousands of different stocks. For the purposes of this project, the essential fields are:
- a timestamp or publication date,
- a headline/title string,
- an associated stock ticker.

The corpus goes beyond Tetlock's single-column design by using a broad asset-linked headline panel. That increases cross-sectional breadth but also changes interpretation: the project is no longer estimating one aggregate media factor for the Dow; it is estimating ticker-specific daily tone from a heterogeneous financial news stream.

#### 2. Daily market data
For each retained ticker, daily price and volume data are used to construct:
- next-day target returns,
- lagged return controls,
- lagged activity controls,
- transaction-cost-aware backtests.

The implementation uses Yahoo Finance to source price data, then performs an explicit quality audit so that delisted or empty ticker shells do not contaminate the investable universe.

#### 3. Lexicon benchmark input
The Loughran–McDonald master dictionary is used to build a finance-lexicon baseline. It is not the primary model, but it is an important benchmark because it represents the older word-list style of sentiment extraction that modern transformers are supposed to improve upon.

### 3.2 Sample construction

The final panel is intentionally narrower than the raw files. The notebook:
- restricts the date range to a fixed sample window,
- requires a minimum amount of news per ticker,
- caps the universe size to keep the project computationally manageable,
- removes tickers with unusable price histories,
- aligns day-\(t\) features with day-\(t+1\) outcomes.


### 3.3 Empirical unit and alignment logic

The final modeling unit is **ticker-day**. Each ticker-day observation is built in four steps:

1. score individual headlines using one or more sentiment extraction methods,
2. aggregate article-level scores to the asset-day level,
3. merge those asset-day indicators with lagged market controls,
4. assign the next-day return and next-day directional label.

This alignment is designed to respect the information set available at decision time. The entire project assumes that daily sentiment is finalized after day \(t\) ends and is used to forecast the next tradable return window on day \(t+1\). That timing convention is central to avoiding leakage in text-based forecasting problems.

### 3.4 Dataset size

After filtering tickers from the raw news dataset, we keep the 40 tickers with the most news coverage, resulting in a dataset of over 100,000 news headlines from 2009 to 2020. This is an average of 2,500 news articles per ticker.

In [ ]:
def load_news_data(config: Dict) -> pd.DataFrame:
    news_path = config["news_data_path"]
    if news_path is None:
        raise FileNotFoundError(
            "No news dataset found. Set CONFIG['news_data_path'] to your CSV path."
        )

    df = pd.read_csv(news_path)
    print(f"Loaded news data: {news_path}")
    print(df.shape)
    print(df.columns.tolist())

    date_col = infer_column(df, ["date", "published_at", "datetime", "timestamp"])
    text_col = infer_column(df, ["headline", "title", "text", "summary"])
    ticker_col = infer_column(df, ["stock", "ticker", "symbol"], required=False)

    out = df.copy()
    out["date"] = pd.to_datetime(out[date_col], errors="coerce", utc=True)
    out["headline"] = out[text_col].astype(str).map(clean_text)

    if ticker_col is not None:
        out["ticker"] = out[ticker_col].map(normalize_ticker)
    else:
        out["ticker"] = out["headline"].str.extract(r"\$([A-Z]{1,5})", expand=False).map(normalize_ticker)

    out = out.dropna(subset=["date", "headline", "ticker"]).copy()
    out["calendar_date"] = out["date"].dt.tz_convert("UTC").dt.date.astype(str)
    out = out.drop_duplicates(subset=["date", "headline", "ticker"]).copy()

    start = pd.Timestamp(config["start_date"], tz="UTC")
    end = pd.Timestamp(config["end_date"], tz="UTC") + pd.Timedelta(days=1)
    out = out[(out["date"] >= start) & (out["date"] < end)].copy()

    coverage = (
        out.groupby("ticker")
           .agg(news_count=("headline", "size"),
                first_date=("date", "min"),
                last_date=("date", "max"))
           .sort_values("news_count", ascending=False)
    )
    eligible = coverage[coverage["news_count"] >= config["min_news_per_ticker"]].head(config["max_tickers"]).index
    out = out[out["ticker"].isin(eligible)].copy()

    print(f"Final news panel shape: {out.shape}")
    print(f"Tickers kept: {out['ticker'].nunique()}")
    display(coverage.head(15))
    return out, coverage.reset_index()

news_df, ticker_coverage = load_news_data(CONFIG)
save_csv(ticker_coverage, "ticker_coverage.csv")
news_df.head()

### 3.5 Exploratory data diagnostics

These plots establish whether the sample is economically meaningful: we want to understand how news coverage evolves over time, whether the panel is concentrated in a few names, and whether the cross section is broad enough to support a daily ranking-based strategy.

As the plots show, the overall news coverage does increase over time (as we would expect) but is still significant enough in early years to use for a predictive model. Many of the top tickers have a few thousand relevant news articles (as previously mentioned), and nearly every one has data from 1000-1500 trading days.

In [ ]:
daily_counts = (
    news_df.groupby(pd.to_datetime(news_df["calendar_date"]))["headline"]
    .size()
    .rename("article_count")
)

top_tickers = news_df["ticker"].value_counts().head(20)

fig, ax = plt.subplots(1, 2, figsize=(16, 5))

daily_counts.plot(ax=ax[0], title="Daily article count")
ax[0].set_ylabel("Articles")

top_tickers.sort_values().plot(kind="barh", ax=ax[1], title="Top tickers by article count")
ax[1].set_xlabel("Articles")

plt.tight_layout()
plt.show()

coverage_summary = (
    news_df.groupby("ticker")
    .agg(
        n_articles=("headline", "size"),
        n_days=("calendar_date", "nunique")
    )
    .sort_values("n_articles", ascending=False)
)

coverage_summary.head(15)

## 4. Indicators & Relevant Tests

Indicator analysis: do the raw text variables exhibit the qualitative behavior one would expect if they are capturing tone, attention, or sentiment pressure?

### 4.1 Indicator families

The notebook works with four conceptually distinct indicator families:

1. **Directional tone** (`sentiment_mean`, `pessimism_mean`)  
   Measures whether the day's news is relatively positive or negative.

2. **Intensity / disagreement** (`abs_sentiment_mean`, `pessimism_std`)  
   Measures whether the day's tone is unusually strong or internally dispersed.

3. **Coverage / attention** (`news_count`, `log_news_count`)  
   Captures how much news attention a ticker received.

4. **Normalized extremeness** (`pessimism_x_news`, `pess_z_20`)  
   Distinguishes ordinary negative tone from unusually negative tone relative to recent history.

### 4.2 Indicator extraction methods

- **FinBERT:** the primary finance-domain sentiment transformer model.
- **SiEBERT:** a generic transformer benchmark.
- **Loughran–McDonald lexicon:** a simple finance-specific word-list benchmark.

Together, these provide one domain-aware model, one generic modern model, and one classical benchmark. That comparison is central to the project's contribution.

### 4.3 Indicator hypothesis and primary tests

The indicator layer is evaluated with **two primary tests**.

1. **Return-screening test.**  
   Does more pessimistic tone line up with weaker next-day returns, either in pooled association or in a coarse bucket sort? This is deliberately weak: indicator-level tests are not expected to produce a complete trading rule.

2. **Activity-channel test.**  
   Does stronger absolute tone predict higher next-day trading activity? This mechanism matters because Tetlock-style theories often imply that extreme tone moves attention and temporary price pressure even when unconditional return effects are small.

In [ ]:
class TransformerSentimentScorer:
    def __init__(self, model_name: str, label_mode: str, batch_size: int = 32, max_length: int = 96):
        if AutoTokenizer is None or AutoModelForSequenceClassification is None:
            raise ImportError("transformers/torch not available in this environment.")

        self.model_name = model_name
        self.label_mode = label_mode
        self.batch_size = batch_size
        self.max_length = max_length

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.config = AutoConfig.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE)
        self.model.eval()

    def _convert_probs(self, probs: np.ndarray) -> pd.DataFrame:
        if self.label_mode == "finbert_3class":
            labels = {int(k): str(v).lower() for k, v in self.config.id2label.items()}
            label_to_index = {label: idx for idx, label in labels.items()}
            pos = probs[:, label_to_index.get("positive", 0)]
            neg = probs[:, label_to_index.get("negative", 1 if probs.shape[1] > 1 else 0)]
            neu = probs[:, label_to_index.get("neutral", 2 if probs.shape[1] > 2 else 0)]
            out = pd.DataFrame({
                "positive_prob": pos,
                "negative_prob": neg,
                "neutral_prob": neu
            })
        elif self.label_mode == "binary_posneg":
            labels = {int(k): str(v).lower() for k, v in self.config.id2label.items()}
            positive_idx = None
            negative_idx = None
            for idx, label in labels.items():
                if "pos" in label:
                    positive_idx = idx
                if "neg" in label:
                    negative_idx = idx
            if positive_idx is None or negative_idx is None:
                positive_idx, negative_idx = 1, 0

            pos = probs[:, positive_idx]
            neg = probs[:, negative_idx]
            out = pd.DataFrame({
                "positive_prob": pos,
                "negative_prob": neg,
                "neutral_prob": np.zeros(len(probs))
            })
        else:
            raise ValueError(f"Unknown label_mode: {self.label_mode}")

        out["sentiment_score"] = out["positive_prob"] - out["negative_prob"]
        out["pessimism"] = out["negative_prob"]
        out["abs_sentiment"] = out["sentiment_score"].abs()
        return out

    @torch.no_grad()
    def score_texts(self, texts: List[str]) -> pd.DataFrame:
        chunks = []
        for i in tqdm(range(0, len(texts), self.batch_size), desc=f"Scoring with {self.model_name}"):
            batch = texts[i:i + self.batch_size]
            encoded = self.tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=self.max_length
            )
            encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
            logits = self.model(**encoded).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            chunks.append(self._convert_probs(probs))
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        return pd.concat(chunks, axis=0, ignore_index=True)

def load_negative_lexicon(path: Optional[str] = None) -> set:
    if path.lower().endswith(".csv"):
        lex_df = pd.read_csv(path)
    else:
        lex_df = pd.read_csv(path, sep=None, engine="python")

    lex_df.columns = [c.strip() for c in lex_df.columns]

    if "Word" in lex_df.columns and "Negative" in lex_df.columns:
        negative_words = set(
            lex_df.loc[lex_df["Negative"].fillna(0) > 0, "Word"]
            .astype(str)
            .str.lower()
            .str.strip()
        )
        return negative_words

    first_col = lex_df.columns[0]
    words = set(
        lex_df[first_col]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
    )
    return words

def lexicon_sentiment_scores(texts: List[str], negative_words: set) -> pd.DataFrame:
    records = []
    for text in tqdm(texts, desc="Scoring lexicon baseline"):
        tokens = re.findall(r"[A-Za-z']+", text.lower())
        n = len(tokens)
        neg = sum(t in negative_words for t in tokens)
        neg_ratio = neg / n if n else 0.0
        records.append({
            "positive_prob": 0.0,
            "negative_prob": neg_ratio,
            "neutral_prob": 0.0,
            "sentiment_score": -neg_ratio,
            "pessimism": neg_ratio,
            "abs_sentiment": abs(neg_ratio)
        })
    return pd.DataFrame(records)

### 4.4 Article-level indicator construction

Each article/headline is scored independently. This preserves a clean separation between:
- raw textual measurement (indicator stage), and
- downstream forecasting / trading decisions (later sections which may aggregate many sentiment factors).

Saving article-level scores is important for reproducibility and later diagnostic work.

In [ ]:
scored_news = news_df.copy().reset_index(drop=True)

if CONFIG["run_finbert"]:
    finbert_scorer = TransformerSentimentScorer(
        model_name="ProsusAI/finbert",
        label_mode="finbert_3class",
        batch_size=CONFIG["transformer_batch_size"],
        max_length=CONFIG["transformer_max_length"],
    )
    finbert_scores = finbert_scorer.score_texts(scored_news["headline"].tolist())
    finbert_scores = finbert_scores.add_prefix("finbert_")
    scored_news = pd.concat([scored_news, finbert_scores], axis=1)

if CONFIG["run_siebert"]:
    siebert_scorer = TransformerSentimentScorer(
        model_name="siebert/sentiment-roberta-large-english",
        label_mode="binary_posneg",
        batch_size=CONFIG["transformer_batch_size"],
        max_length=CONFIG["transformer_max_length"],
    )
    siebert_scores = siebert_scorer.score_texts(scored_news["headline"].tolist())
    siebert_scores = siebert_scores.add_prefix("siebert_")
    scored_news = pd.concat([scored_news, siebert_scores], axis=1)

if CONFIG["run_lexicon"]:
    neg_words = load_negative_lexicon(CONFIG["lm_negative_words_path"])
    lex_scores = lexicon_sentiment_scores(scored_news["headline"].tolist(), neg_words)
    lex_scores = lex_scores.add_prefix("lex_")
    scored_news = pd.concat([scored_news, lex_scores], axis=1)

print(scored_news.shape)
scored_news.head()

In [ ]:
save_csv(scored_news, "article_level_sentiment_scores.csv")

### 4.5 Aggregate article indicators to the asset-day level

Trading decisions are formed at the **asset-day** level, so article-level measurements must be aggregated into a daily panel. The daily statistics below are the core building blocks used in later signal and rule sections.

In [ ]:
def aggregate_daily_panel(scored_news: pd.DataFrame, prefix: str) -> pd.DataFrame:
    cols = {
        "pessimism": f"{prefix}_pessimism",
        "sentiment_score": f"{prefix}_sentiment_score",
        "abs_sentiment": f"{prefix}_abs_sentiment",
    }
    required = list(cols.values())
    missing = [c for c in required if c not in scored_news.columns]
    if missing:
        raise KeyError(f"Missing columns for prefix '{prefix}': {missing}")

    g = (
        scored_news.groupby(["calendar_date", "ticker"])
        .agg(
            news_count=("headline", "size"),
            pessimism_mean=(cols["pessimism"], "mean"),
            pessimism_std=(cols["pessimism"], "std"),
            sentiment_mean=(cols["sentiment_score"], "mean"),
            abs_sentiment_mean=(cols["abs_sentiment"], "mean"),
            pessimism_min=(cols["pessimism"], "min"),
            pessimism_max=(cols["pessimism"], "max"),
        )
        .reset_index()
    )

    g["calendar_date"] = pd.to_datetime(g["calendar_date"])
    g["pessimism_std"] = g["pessimism_std"].fillna(0.0)
    return g

panel_dict = {}

if CONFIG["run_finbert"]:
    panel_dict["finbert"] = aggregate_daily_panel(scored_news, "finbert")
if CONFIG["run_siebert"]:
    panel_dict["siebert"] = aggregate_daily_panel(scored_news, "siebert")
if CONFIG["run_lexicon"]:
    panel_dict["lex"] = aggregate_daily_panel(scored_news, "lex")

for k, v in panel_dict.items():
    print(k, v.shape)
    save_csv(v, f"daily_panel_{k}.csv")

### 4.6 Align indicators with outcomes and market controls

Once the asset-day indicators are constructed, they must be aligned to future outcomes and standard market controls. This step determines whether the later statistical tests are valid.

The design principle is simple: all variables available at the close of day \(t\) may be used to forecast a return realized on day \(t+1\). The panel therefore includes:

- lagged return controls,
- lagged trading-activity controls,
- simple seasonal dummies,
- day-\(t\) sentiment indicators, and
- a day-\(t+1\) target return and direction label.

The code also audits the price universe before proceeding. Securities with missing or unusable price histories are removed so that the modeling panel reflects the true investable universe rather than the raw set of ticker labels present in the news file.



In [ ]:
def load_price_data_for_tickers(tickers: List[str], config: Dict) -> pd.DataFrame:
    if yf is None:
        raise ImportError("yfinance is not available and no local price_data_path was provided.")
    px = yf.download(
        tickers=tickers,
        start=config["start_date"],
        end=(pd.Timestamp(config["end_date"]) + pd.Timedelta(days=7)).strftime("%Y-%m-%d"),
        auto_adjust=False,
        progress=True,
        group_by="ticker",
        threads=True
    )
    print("Downloaded price data with yfinance.")
    return px


def prices_to_long(px, tickers: List[str]) -> pd.DataFrame:
    if isinstance(px.columns, pd.MultiIndex):
        frames = []
        for ticker in tickers:
            if ticker not in px.columns.get_level_values(0):
                continue

            tmp = px[ticker].copy().reset_index()
            tmp.columns = [c.lower() if isinstance(c, str) else c for c in tmp.columns]
            tmp["ticker"] = ticker

            core_cols = [c for c in ["open", "close", "volume"] if c in tmp.columns]
            if core_cols and tmp[core_cols].notna().sum().sum() == 0:
                print(f"Skipping {ticker}: no usable price data returned.")
                continue

            frames.append(tmp)

        if not frames:
            raise ValueError("No usable price data found for any ticker.")

        out = pd.concat(frames, axis=0, ignore_index=True)

    else:
        out = px.copy().reset_index()
        out.columns = [c.lower() if isinstance(c, str) else c for c in out.columns]
        if "ticker" not in out.columns:
            raise ValueError("Price file must contain ticker column if not using multi-index yfinance format.")

    if "date" not in out.columns:
        date_col = infer_column(out, ["date", "datetime"])
        out["date"] = pd.to_datetime(out[date_col], errors="coerce")
    else:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")

    out["calendar_date"] = out["date"].dt.date.astype(str)

    rename_map = {}
    for c in ["open", "high", "low", "close", "adj close", "volume"]:
        if c in out.columns:
            rename_map[c] = c.replace(" ", "_")
    out = out.rename(columns=rename_map)

    return out


def build_return_panel(price_long: pd.DataFrame, target_return_type: str = "next_open_to_close") -> pd.DataFrame:
    df = price_long.sort_values(["ticker", "date"]).copy()

    for col in ["open", "close", "volume"]:
        if col not in df.columns:
            raise KeyError(f"Price data must include column '{col}'")

    df["close_to_close_ret"] = df.groupby("ticker")["close"].transform(
        lambda s: safe_log_return(s, s.shift(1))
    )
    df["open_to_close_ret"] = safe_log_return(df["close"], df["open"])

    df["next_open_to_close_ret"] = df.groupby("ticker")["open_to_close_ret"].shift(-1)
    df["next_close_to_close_ret"] = df.groupby("ticker")["close_to_close_ret"].shift(-1)

    df["ret_lag_1"] = df.groupby("ticker")["close_to_close_ret"].shift(1)
    df["ret_lag_2"] = df.groupby("ticker")["close_to_close_ret"].shift(2)
    df["ret_lag_3"] = df.groupby("ticker")["close_to_close_ret"].shift(3)
    df["ret_lag_4"] = df.groupby("ticker")["close_to_close_ret"].shift(4)
    df["ret_lag_5"] = df.groupby("ticker")["close_to_close_ret"].shift(5)

    df["vol_lag_1"] = df.groupby("ticker")["volume"].shift(1)
    df["log_vol_lag_1"] = np.log(df["vol_lag_1"].where(df["vol_lag_1"] > 0))
    df["realized_abs_ret_lag_1"] = df.groupby("ticker")["close_to_close_ret"].shift(1).abs()

    if target_return_type == "next_open_to_close":
        df["target_return"] = df["next_open_to_close_ret"]
    elif target_return_type == "next_close_to_close":
        df["target_return"] = df["next_close_to_close_ret"]
    else:
        raise ValueError("Unknown target_return_type")

    df["target_up"] = (df["target_return"] > 0).astype(float)
    df["next_log_volume"] = np.log(df.groupby("ticker")["volume"].shift(-1).where(lambda s: s > 0))
    df["weekday"] = pd.to_datetime(df["calendar_date"]).dt.dayofweek
    df["month"] = pd.to_datetime(df["calendar_date"]).dt.month
    df["january_dummy"] = (df["month"] == 1).astype(int)

    return df


def audit_and_filter_price_universe(
    return_panel: pd.DataFrame,
    news_df: pd.DataFrame,
    min_valid_target_days: int = 252
):
    price_audit = (
        return_panel.groupby("ticker")
        .agg(
            rows=("date", "size"),
            nonnull_open=("open", lambda s: s.notna().sum()),
            nonnull_close=("close", lambda s: s.notna().sum()),
            nonnull_volume=("volume", lambda s: s.notna().sum()),
            target_nonnull=("target_return", lambda s: s.notna().sum())
        )
        .sort_values("target_nonnull", ascending=False)
    )

    valid_price_tickers = price_audit.loc[
        price_audit["target_nonnull"] >= min_valid_target_days
    ].index.tolist()

    dropped_price_tickers = price_audit.loc[
        price_audit["target_nonnull"] < min_valid_target_days
    ].index.tolist()

    print(f"Tickers with usable price history (>= {min_valid_target_days} valid target-return days): {len(valid_price_tickers)}")
    print(f"Dropped tickers: {dropped_price_tickers}")

    filtered_return_panel = return_panel[return_panel["ticker"].isin(valid_price_tickers)].copy()
    filtered_news_df = news_df[news_df["ticker"].isin(valid_price_tickers)].copy()

    print("Filtered return panel shape:", filtered_return_panel.shape)
    print("Filtered news panel shape:", filtered_news_df.shape)

    return filtered_return_panel, filtered_news_df, price_audit.reset_index()


tickers = sorted(news_df["ticker"].dropna().unique().tolist())
price_raw = load_price_data_for_tickers(tickers, CONFIG)
price_long = prices_to_long(price_raw, tickers)
return_panel = build_return_panel(price_long, CONFIG["target_return_type"])

return_panel, news_df, price_audit = audit_and_filter_price_universe(
    return_panel=return_panel,
    news_df=news_df,
    min_valid_target_days=252
)

save_csv(price_audit, "price_audit.csv")

print(return_panel.shape)
display(price_audit.head(20))
return_panel.head()

In [ ]:
save_csv(return_panel, "price_return_panel.csv")

### 4.7 Modeling panel construction

This merged panel is the core research dataset used in all later sections. Each row corresponds to one ticker on one date and contains three conceptually distinct blocks of information:

1. **Indicators:** asset-day sentiment summaries derived from text.
2. **Controls:** lagged returns, lagged volume, and related market-state variables.
3. **Outcomes:** next-day return and next-day up/down indicator.

This separation is deliberate. It allows the notebook to test whether sentiment matters by itself, whether standard market variables dominate, and whether sentiment adds incremental value once both are included together. The merge also creates interaction and rolling-normalization variables used later in the signal process.



In [ ]:
def merge_sentiment_with_returns(daily_sentiment: pd.DataFrame, return_panel: pd.DataFrame, model_name: str) -> pd.DataFrame:
    left = daily_sentiment.copy()
    right = return_panel.copy()

    left["calendar_date"] = pd.to_datetime(left["calendar_date"], errors="coerce").dt.strftime("%Y-%m-%d")
    right["calendar_date"] = pd.to_datetime(right["calendar_date"], errors="coerce").dt.strftime("%Y-%m-%d")

    left["ticker"] = left["ticker"].astype(str).str.upper().str.strip()
    right["ticker"] = right["ticker"].astype(str).str.upper().str.strip()

    merged = left.merge(
        right,
        on=["calendar_date", "ticker"],
        how="inner"
    ).copy()

    merged["log_news_count"] = np.log1p(merged["news_count"])
    merged["pessimism_x_news"] = merged["pessimism_mean"] * merged["log_news_count"]

    merged = merged.sort_values(["ticker", "calendar_date"]).copy()
    lagged_pess = merged.groupby("ticker")["pessimism_mean"].shift(1)

    merged["pess_rolling_mean_20"] = (
        lagged_pess.groupby(merged["ticker"])
        .transform(lambda s: s.rolling(20, min_periods=5).mean())
    )
    merged["pess_rolling_std_20"] = (
        lagged_pess.groupby(merged["ticker"])
        .transform(lambda s: s.rolling(20, min_periods=5).std())
    )

    merged["pess_z_20"] = (
        (merged["pessimism_mean"] - merged["pess_rolling_mean_20"]) /
        merged["pess_rolling_std_20"]
    )

    merged["source_model"] = model_name
    return merged

model_panels = {name: merge_sentiment_with_returns(panel, return_panel, name) for name, panel in panel_dict.items()}

for name, df in model_panels.items():
    print(name, df.shape)
    save_csv(df, f"model_panel_{name}.csv")



### 4.8 Indicator-only tests

Before fitting any forecasting model, the notebook tests the indicators directly. This is a necessary diagnostic step. If raw sentiment has no economically sensible relation to returns, then a later positive backtest may be more plausibly attributed to model flexibility or chance than to the intended sentiment mechanism.

The indicator tests are deliberately simple:

- pooled correlations between pessimism and next-day return,
- pooled correlations between sentiment magnitude and next-day volume,
- return bucket sorts by pessimism level.

These are not intended to be definitive causal tests. Instead, they provide a first-pass check on sign, monotonicity, and relative strength across sentiment sources.



In [ ]:
def pooled_simple_stats(panel: pd.DataFrame, title: str):
    tmp = panel[["pessimism_mean", "target_return", "abs_sentiment_mean", "next_log_volume"]].dropna().copy()
    if len(tmp) < 50:
        print(f"Not enough observations for {title}")
        return
    corr1 = tmp["pessimism_mean"].corr(tmp["target_return"])
    corr2 = tmp["abs_sentiment_mean"].corr(tmp["next_log_volume"])
    print(f"{title}: corr(pessimism, next return) = {corr1:.4f}")
    print(f"{title}: corr(abs sentiment, next log volume) = {corr2:.4f}")

for name, df in model_panels.items():
    pooled_simple_stats(df, name)

In [ ]:
primary_name = "finbert" if "finbert" in model_panels else list(model_panels.keys())[0]
primary_panel = model_panels[primary_name].copy()

plot_df = primary_panel.dropna(subset=["pessimism_mean", "target_return"]).copy()
plot_df["pess_bucket"] = pd.qcut(plot_df["pessimism_mean"], q=5, labels=False, duplicates="drop")

bucket_ret = plot_df.groupby("pess_bucket")["target_return"].mean()

bucket_ret.plot(kind="bar", title=f"Average next-day return by {primary_name} pessimism quintile")
plt.axhline(0, color="black", linewidth=1)
plt.ylabel("Mean next-day log return")
plt.show()

### 4.9 Indicator-test analysis

At the indicator level, the evidence is weak but directionally plausible rather than decisively predictive. The pooled correlations between pessimism and next-day returns are quite close to zero for all three sentiment constructions, which indicates that raw sentiment on its own has very limited unconditional return-forecasting power in this daily panel. The FinBERT quintile plot is consistent with that conclusion: more pessimistic news is generally associated with more negative subsequent returns, but the pattern is not cleanly monotonic and the economic magnitudes remain small, suggesting that any return-related information in the raw indicator is noisy and likely conditional on other controls rather than directly tradable by itself. By contrast, the activity-channel results provide somewhat more support for the underlying mechanism. Absolute sentiment is positively related to next-day log volume for all models, with the strongest association coming from FinBERT, which is consistent with a Tetlock-style interpretation that emotionally intense or salient news is more reliably linked to investor attention and trading activity than to immediate directional price moves. Taken together, these results suggest that the raw indicator layer is not strong enough to justify a standalone trading rule, but it does appear to contain a modest and economically sensible signal—especially for finance-specific sentiment—that may become more useful once embedded in the later signal-generation and rule-construction stages.

## 5. Signal process & Relevant Tests

The signal process converts raw indicators into an explicit **forecast** that can later be mapped into positions. In this study, the core signal is the estimated probability that a stock's day-$t+1$ return is positive. This layer is evaluated separately from the trading rule because the forecast itself must be shown to add value before any portfolio construction claims are made.

### 5.1 Signal definition

Let $X_{i,t}$ denote the feature vector for stock $i$ on day $t$. The signal process estimates

$
\hat p_{i,t} = \Pr(r_{i,t+1} > 0 \mid X_{i,t}),
$

where $r_{i,t+1}$ is the next-day target return. The fitted probability $\hat p_{i,t}$ is used in two ways:

1. as a **directional probability forecast**, and
2. as a **continuous ranking score** for cross-sectional sorting.

### 5.2 Signal hypothesis and primary tests

We hypothesize that sentiment should add incremental predictive information after conditioning on lagged market controls. This section is intentionally streamlined to **two primary tests**.

1. **Forecast-quality test.**  
   Compare `baseline_market_only` and `sentiment_plus_controls` out of sample. The key question is whether adding sentiment improves probability/discrimination metrics in a time-respecting setting.

2. **Ranking-value test.**  
   Compare the realized next-day returns of the highest-scored versus lowest-scored observations. This tests whether the signal has cross-sectional economic ordering value before any trading rule is imposed.

Calibration output is retained as a **diagnostic**.

### 5.3 Why simple models?

The primary signal model is logistic regression, which is deliberate. A linear, regularized classifier is transparent, stable, easy to audit, and well matched to the project's goal of showing whether sentiment adds incremental information beyond standard controls. A higher-capacity boosted-tree model is kept as an extension.

### 5.4 Feature design

The signal section uses three nested feature sets:

- **`baseline_market_only`**: lagged return, volume, and calendar controls.
- **`sentiment_only`**: raw text-derived variables without market controls.
- **`sentiment_plus_controls`**: the combined specification.

This is crucial for identification: for example, if sentiment-only performs weakly but sentiment-plus-controls improves meaningfully on the baseline, the correct conclusion is that text contributes conditional information rather than a standalone signal.

In [ ]:
from sklearn.metrics import log_loss, brier_score_loss

BASE_CONTROL_FEATURES = [
    "ret_lag_1", "ret_lag_2", "ret_lag_3", "ret_lag_4", "ret_lag_5",
    "log_vol_lag_1", "realized_abs_ret_lag_1", "january_dummy"
]

SENTIMENT_FEATURES = [
    "pessimism_mean", "pessimism_std", "sentiment_mean",
    "abs_sentiment_mean", "news_count", "log_news_count",
    "pessimism_x_news", "pess_z_20"
]

FEATURE_SETS = {
    "baseline_market_only": BASE_CONTROL_FEATURES,
    "sentiment_only": ["pessimism_mean", "sentiment_mean", "abs_sentiment_mean", "news_count"],
    "sentiment_plus_controls": BASE_CONTROL_FEATURES + SENTIMENT_FEATURES,
}

SIGNAL_MODEL_FACTORIES = {
    "logistic": lambda: Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced"))
    ])
}

if HAS_XGB:
    SIGNAL_MODEL_FACTORIES["xgboost"] = lambda: XGBClassifier(
        n_estimators=150,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        min_child_weight=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
    )

def make_logistic_model():
    return SIGNAL_MODEL_FACTORIES["logistic"]()

def make_xgb_model():
    if HAS_XGB:
        return SIGNAL_MODEL_FACTORIES["xgboost"]()
    return make_logistic_model()

### 5.5 Signal-process test design

The signal layer is evaluated with walk-forward tests that stop short of the portfolio rule.

#### Primary Test 1: forecast-quality comparison
For each sentiment source and feature set, the notebook estimates the signal out of sample and summarizes forecast quality using a compact set of metrics:
- **AUC** as the main discrimination metric,
- **log loss** as the main probability-quality metric,
- and balanced accuracy as a secondary descriptive statistic.

The central comparison is whether `sentiment_plus_controls` improves on `baseline_market_only`.

#### Primary Test 2: ranking-value comparison
The same predicted probabilities are then treated as ranking scores. The notebook asks whether high-score observations outperform low-score observations in next-day returns through a top-minus-bottom score spread. This is the cleanest signal-level bridge from statistical forecasting to later trading.

#### Diagnostic
A calibration table and plot are retained for the best signal specification. They help determine whether the estimated probabilities are sensible or merely noisy scores.

In [ ]:
def make_signal_process_folds(panel: pd.DataFrame, train_min_days: int, test_days: int, step_days: Optional[int] = None):
    df = panel.copy()
    df["calendar_date"] = pd.to_datetime(df["calendar_date"])
    dates = sorted(df["calendar_date"].dropna().unique().tolist())
    if step_days is None:
        step_days = test_days

    folds = []
    start_test = train_min_days
    while start_test + test_days <= len(dates):
        train_dates = set(dates[:start_test])
        test_dates = set(dates[start_test:start_test + test_days])
        folds.append((train_dates, test_dates))
        start_test += step_days
    return folds


def evaluate_signal_predictions(pred_df: pd.DataFrame) -> Dict[str, float]:
    tmp = pred_df[["pred_proba", "pred_label", "target_up", "target_return"]].dropna().copy()
    if len(tmp) < 100:
        return {
            "n_obs": len(tmp),
            "accuracy": np.nan,
            "balanced_accuracy": np.nan,
            "auc": np.nan,
            "log_loss": np.nan,
            "brier": np.nan,
            "spearman_score_vs_return": np.nan,
            "top_minus_bottom_ret": np.nan,
        }

    y_true = tmp["target_up"].astype(int).values
    y_pred = tmp["pred_label"].astype(int).values
    y_prob = np.clip(tmp["pred_proba"].astype(float).values, 1e-6, 1 - 1e-6)

    try:
        auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    except Exception:
        auc = np.nan

    try:
        ll = log_loss(y_true, y_prob)
    except Exception:
        ll = np.nan

    try:
        brier = brier_score_loss(y_true, y_prob)
    except Exception:
        brier = np.nan

    try:
        spear = stats.spearmanr(tmp["pred_proba"], tmp["target_return"], nan_policy="omit").correlation
    except Exception:
        spear = np.nan

    tmp["score_bucket"] = pd.qcut(tmp["pred_proba"], 5, labels=False, duplicates="drop")
    bucket_means = tmp.groupby("score_bucket")["target_return"].mean()
    spread = bucket_means.iloc[-1] - bucket_means.iloc[0] if len(bucket_means) >= 2 else np.nan

    return {
        "n_obs": len(tmp),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "auc": auc,
        "log_loss": ll,
        "brier": brier,
        "spearman_score_vs_return": spear,
        "top_minus_bottom_ret": spread,
    }


def run_signal_process_test(panel: pd.DataFrame, feature_cols: List[str], model_factory, model_name: str,
                            train_min_days: int, test_days: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    cols = ["calendar_date", "ticker", "target_up", "target_return"] + feature_cols
    data = panel[cols].dropna(subset=["target_up", "target_return"]).copy()
    data["calendar_date"] = pd.to_datetime(data["calendar_date"])
    data = data.sort_values(["calendar_date", "ticker"]).copy()

    folds = make_signal_process_folds(data, train_min_days=train_min_days, test_days=test_days)
    fold_rows = []
    pred_frames = []

    for fold_idx, (train_dates, test_dates) in enumerate(folds, start=1):
        train_df = data[data["calendar_date"].isin(train_dates)].copy()
        test_df = data[data["calendar_date"].isin(test_dates)].copy()
        if len(train_df) < 1000 or len(test_df) < 100:
            continue

        X_train = train_df[feature_cols]
        y_train = train_df["target_up"].astype(int)
        X_test = test_df[feature_cols]
        y_test = test_df["target_up"].astype(int)

        model = model_factory()
        model.fit(X_train, y_train)

        if hasattr(model, "predict_proba"):
            pred_proba = model.predict_proba(X_test)[:, 1]
        else:
            raw_score = model.decision_function(X_test)
            pred_proba = 1 / (1 + np.exp(-raw_score))

        pred_df = test_df[["calendar_date", "ticker", "target_up", "target_return"]].copy()
        pred_df["pred_proba"] = pred_proba
        pred_df["pred_label"] = (pred_df["pred_proba"] >= 0.5).astype(int)
        pred_df["fold"] = fold_idx
        pred_df["signal_model"] = model_name
        pred_frames.append(pred_df)

        metrics = evaluate_signal_predictions(pred_df)
        metrics["fold"] = fold_idx
        metrics["train_start"] = min(train_dates) if len(train_dates) else pd.NaT
        metrics["train_end"] = max(train_dates) if len(train_dates) else pd.NaT
        metrics["test_start"] = min(test_dates) if len(test_dates) else pd.NaT
        metrics["test_end"] = max(test_dates) if len(test_dates) else pd.NaT
        fold_rows.append(metrics)

    return pd.DataFrame(fold_rows), pd.concat(pred_frames, axis=0, ignore_index=True) if pred_frames else pd.DataFrame()


def make_calibration_table(pred_df: pd.DataFrame, n_bins: int = 10) -> pd.DataFrame:
    tmp = pred_df[["pred_proba", "target_up"]].dropna().copy()
    if len(tmp) < 100:
        return pd.DataFrame()
    tmp["prob_bin"] = pd.qcut(tmp["pred_proba"], q=n_bins, duplicates="drop")
    out = tmp.groupby("prob_bin").agg(
        mean_pred_prob=("pred_proba", "mean"),
        realized_up_rate=("target_up", "mean"),
        n_obs=("target_up", "size")
    ).reset_index()
    out["calibration_gap"] = out["realized_up_rate"] - out["mean_pred_prob"]
    return out



In [ ]:
signal_process_rows = []
signal_process_predictions = {}
signal_process_fold_metrics = {}

for sentiment_source, panel in model_panels.items():
    for feature_set_name, feature_cols in FEATURE_SETS.items():
        for signal_model_name, model_factory in SIGNAL_MODEL_FACTORIES.items():
            if signal_model_name == "xgboost" and feature_set_name != "sentiment_plus_controls":
                continue
            print(f"Signal test -> source={sentiment_source} | features={feature_set_name} | model={signal_model_name}")
            try:
                fold_df, pred_df = run_signal_process_test(
                    panel=panel,
                    feature_cols=feature_cols,
                    model_factory=model_factory,
                    model_name=signal_model_name,
                    train_min_days=CONFIG["train_min_days"],
                    test_days=CONFIG["test_days"],
                )
                if fold_df.empty or pred_df.empty:
                    continue

                fold_df["sentiment_source"] = sentiment_source
                fold_df["feature_set"] = feature_set_name
                fold_df["signal_model"] = signal_model_name
                signal_process_fold_metrics[(sentiment_source, feature_set_name, signal_model_name)] = fold_df
                signal_process_predictions[(sentiment_source, feature_set_name, signal_model_name)] = pred_df

                summary = fold_df[[
                    "accuracy", "balanced_accuracy", "auc", "log_loss", "brier",
                    "spearman_score_vs_return", "top_minus_bottom_ret"
                ]].mean().to_dict()
                summary.update({
                    "sentiment_source": sentiment_source,
                    "feature_set": feature_set_name,
                    "signal_model": signal_model_name,
                    "n_folds": int(fold_df["fold"].nunique()),
                    "n_predictions": int(len(pred_df)),
                })
                signal_process_rows.append(summary)
            except Exception as e:
                print(f"Skipped signal test due to error: {e}")

signal_process_results = pd.DataFrame(signal_process_rows)
if not signal_process_results.empty:
    signal_process_results = signal_process_results.sort_values(
        ["auc", "top_minus_bottom_ret", "balanced_accuracy"], ascending=False
    ).reset_index(drop=True)
    display(signal_process_results)
    save_csv(signal_process_results, "signal_process_summary_results.csv")



In [ ]:
if not signal_process_results.empty:
    signal_primary_summary = signal_process_results[[
        "sentiment_source", "feature_set", "signal_model",
        "auc", "log_loss", "top_minus_bottom_ret",
        "balanced_accuracy", "n_folds", "n_predictions"
    ]].copy().sort_values(
        ["auc", "top_minus_bottom_ret"], ascending=False
    )
    display(signal_primary_summary)
    save_csv(signal_primary_summary, "signal_process_primary_summary.csv")

    rows = []
    for (sentiment_source, signal_model), g in signal_process_results.groupby(["sentiment_source", "signal_model"]):
        row = {"sentiment_source": sentiment_source, "signal_model": signal_model}
        base = g[g["feature_set"] == "baseline_market_only"]
        full = g[g["feature_set"] == "sentiment_plus_controls"]

        if not base.empty and not full.empty:
            row["delta_auc_full_minus_base"] = float(full["auc"].iloc[0] - base["auc"].iloc[0])
            row["delta_logloss_full_minus_base"] = float(full["log_loss"].iloc[0] - base["log_loss"].iloc[0])
            row["delta_spread_full_minus_base"] = float(full["top_minus_bottom_ret"].iloc[0] - base["top_minus_bottom_ret"].iloc[0])
        rows.append(row)

    signal_incremental_tests = pd.DataFrame(rows)
    display(signal_incremental_tests)
    save_csv(signal_incremental_tests, "signal_process_incremental_tests.csv")
else:
    signal_primary_summary = pd.DataFrame()
    signal_incremental_tests = pd.DataFrame()

if signal_process_predictions and not signal_process_results.empty:
    best_key = sorted(
        signal_process_predictions.keys(),
        key=lambda k: (
            signal_process_results.loc[
                (signal_process_results["sentiment_source"] == k[0]) &
                (signal_process_results["feature_set"] == k[1]) &
                (signal_process_results["signal_model"] == k[2]),
                "auc"
            ].iloc[0]
        ),
        reverse=True
    )[0]

    best_pred_df = signal_process_predictions[best_key].copy()
    calib_df = make_calibration_table(best_pred_df, n_bins=10)
    if not calib_df.empty:
        display(calib_df)
        save_csv(calib_df, "signal_process_best_calibration.csv")

        plt.figure(figsize=(7, 5))
        plt.plot(calib_df["mean_pred_prob"], calib_df["realized_up_rate"], marker="o")
        plt.plot([0, 1], [0, 1], linestyle="--")
        plt.xlabel("Mean predicted probability")
        plt.ylabel("Realized up-rate")
        plt.title(f"Signal calibration diagnostic: {best_key[0]} | {best_key[1]} | {best_key[2]}")
        plt.show()

    rank_plot_df = best_pred_df[["pred_proba", "target_return"]].dropna().copy()
    rank_plot_df["score_bucket"] = pd.qcut(rank_plot_df["pred_proba"], 5, labels=False, duplicates="drop")
    rank_means = rank_plot_df.groupby("score_bucket")["target_return"].mean()

    plt.figure(figsize=(8, 4))
    plt.bar(rank_means.index.astype(str), rank_means.values)
    plt.axhline(0, color="black", linewidth=1)
    plt.xlabel("Predicted-probability bucket")
    plt.ylabel("Mean next-day log return")
    plt.title(f"Signal ranking diagnostic: {best_key[0]} | {best_key[1]} | {best_key[2]}")
    plt.show()

### 5.6 Signal-process test analysis

At the signal-process level, the evidence is modestly supportive but economically narrow, with the clearest gains coming from the simplest sentiment-only logistic specifications rather than from richer sentiment-plus-controls models. In the walk-forward forecast-quality test, the best performer is the SieBERT sentiment-only logistic model, which achieves the highest AUC (0.5096), the best log loss (0.6931), and the strongest balanced accuracy (0.5061), with the lexicon model somewhat weaker and FinBERT weaker still; importantly, all of these improvements over chance are small in absolute magnitude, so the signal should be viewed as statistically detectable rather than strongly predictive. The incremental comparison against the market-only baseline is mixed: adding sentiment to controls slightly improves AUC for the logistic models, but does not improve log loss in an economically meaningful way, and the xgboost variants generally fail to add value, suggesting that the available information is weak and may be better captured by a simpler linear mapping than by a more flexible nonlinear model. The ranking-value test is more encouraging: the best specification produces a positive top-minus-bottom return spread of about 10 bps in log-return terms, and the ranking plot shows that the lowest-score bucket performs materially worse than the highest-score bucket, which indicates that the fitted probabilities do contain some useful cross-sectional ordering even when headline classification metrics remain near 0.50. Finally, the calibration diagnostic shows that predicted probabilities are tightly clustered near 0.5 and realized up-rates track them closely, implying that the model is reasonably calibrated but only over a very narrow probability range. In other words, the signal behaves like a well-behaved but weak forecaster. Overall, these results support the interpretation that sentiment contributes a small but potentially usable forecasting layer, especially as a ranking device, but not a high-confidence directional probability model on its own.

## 6. Rule process & Relevant tests

Note: rule process tests are defined and set up in this section, but carried out in section 8 using the walk-forward prediction stream (analysis included in section 8 as well).

### 6.1 Rule families

#### Primary rule: cross-sectional ranking portfolio
Each day, rank assets by forecast score, go long the top `N`, short the bottom `N`, and equal-weight within each side. This is the main economic expression of the signal and is the preferred rule because it naturally uses the continuous forecast rather than forcing every prediction into a binary decision.

#### Benchmark rule: Tetlock-style rolling-tercile comparator
This rule defines unusually optimistic and pessimistic observations relative to rolling history and trades a simpler long/short signal directly off the indicator. It is intentionally lower complexity and provides a close-to-Tetlock benchmark.

### 6.2 Rule hypothesis and primary tests

This section uses **two primary tests**.

1. **Primary-rule economic test.**  
   Evaluate the ranking portfolio's net performance, turnover, and implementation profile under the chosen cost assumptions.

2. **Benchmark-rule comparison.**  
   Compare the ranking rule to the Tetlock-style comparator to determine whether the forecasting layer adds value beyond a direct indicator rule.

Additional breadth or cost tables should be treated as implementation diagnostics.

In [ ]:
def cross_sectional_long_short(pred_df: pd.DataFrame, cost_bps: float, top_n: int = 5) -> pd.DataFrame:
    df = pred_df.copy()
    df["calendar_date"] = pd.to_datetime(df["calendar_date"])
    out_rows = []
    prev_weights = None

    for dt, g in df.groupby("calendar_date"):
        g = g.sort_values("pred_proba", ascending=False).copy()
        longs = g.head(top_n).copy()
        shorts = g.tail(top_n).copy()

        weights = pd.Series(0.0, index=g["ticker"])
        if len(longs) > 0:
            weights.loc[longs["ticker"]] = 1.0 / max(len(longs), 1)
        if len(shorts) > 0:
            weights.loc[shorts["ticker"]] = -1.0 / max(len(shorts), 1)

        ret_map = g.set_index("ticker")["target_return"]
        gross_ret = float((weights * ret_map.reindex(weights.index).fillna(0)).sum())

        if prev_weights is None:
            turnover = float(weights.abs().sum())
        else:
            aligned_index = prev_weights.index.union(weights.index)
            turnover = float((weights.reindex(aligned_index).fillna(0) - prev_weights.reindex(aligned_index).fillna(0)).abs().sum())

        cost = turnover * (cost_bps / 10000.0) / 2.0
        net_ret = gross_ret - cost

        out_rows.append({
            "calendar_date": dt,
            "gross_return": gross_ret,
            "turnover": turnover,
            "cost": cost,
            "net_return": net_ret,
            "n_longs": int((weights > 0).sum()),
            "n_shorts": int((weights < 0).sum()),
            "gross_exposure": float(weights.abs().sum()),
            "net_exposure": float(weights.sum()),
            "max_weight": float(weights.abs().max()) if len(weights) else np.nan
        })
        prev_weights = weights.copy()

    return pd.DataFrame(out_rows)

def rolling_tercile_signal(panel: pd.DataFrame, lookback: int = 252) -> pd.DataFrame:
    df = panel[["calendar_date", "ticker", "pessimism_mean", "target_return"]].dropna().copy()
    df["calendar_date"] = pd.to_datetime(df["calendar_date"])
    df = df.sort_values(["ticker", "calendar_date"]).copy()

    signal_list = []
    for ticker, g in df.groupby("ticker"):
        g = g.sort_values("calendar_date").copy()
        pess = g["pessimism_mean"]
        low = pess.rolling(lookback, min_periods=60).quantile(1/3)
        high = pess.rolling(lookback, min_periods=60).quantile(2/3)

        sig = pd.Series(0, index=g.index, dtype=float)
        sig[pess <= low] = 1.0
        sig[pess >= high] = -1.0

        g["signal"] = sig
        signal_list.append(g)

    sig_df = pd.concat(signal_list, axis=0, ignore_index=True)
    return sig_df

def equal_weight_signal_backtest(sig_df: pd.DataFrame, cost_bps: float) -> pd.DataFrame:
    df = sig_df.copy()
    out_rows = []
    prev_weights = None

    for dt, g in df.groupby("calendar_date"):
        current = g[g["signal"] != 0].copy()
        if len(current) == 0:
            weights = pd.Series(dtype=float)
            gross_ret = 0.0
        else:
            gross_weight = 1.0 / len(current)
            weights = pd.Series(current["signal"].values * gross_weight, index=current["ticker"])
            gross_ret = float((weights * current.set_index("ticker")["target_return"].reindex(weights.index)).sum())

        if prev_weights is None:
            turnover = float(weights.abs().sum()) if len(weights) else 0.0
        else:
            aligned_index = prev_weights.index.union(weights.index)
            turnover = float((weights.reindex(aligned_index).fillna(0) - prev_weights.reindex(aligned_index).fillna(0)).abs().sum())

        cost = turnover * (cost_bps / 10000.0) / 2.0
        net_ret = gross_ret - cost

        out_rows.append({
            "calendar_date": dt,
            "gross_return": gross_ret,
            "turnover": turnover,
            "cost": cost,
            "net_return": net_ret,
            "gross_exposure": float(weights.abs().sum()) if len(weights) else 0.0,
            "net_exposure": float(weights.sum()) if len(weights) else 0.0,
            "max_weight": float(weights.abs().max()) if len(weights) else np.nan
        })
        prev_weights = weights.copy()

    return pd.DataFrame(out_rows)

def summarize_backtest(bt: pd.DataFrame, label: str) -> Dict[str, float]:
    r = bt["net_return"].fillna(0)
    return {
        "strategy": label,
        "annual_return": annualized_return(r, CONFIG["trading_days_per_year"]),
        "annual_vol": annualized_vol(r, CONFIG["trading_days_per_year"]),
        "sharpe": annualized_sharpe(r, CONFIG["trading_days_per_year"]),
        "max_drawdown": compute_max_drawdown(r),
        "avg_turnover": float(bt["turnover"].mean()),
        "avg_gross_exposure": float(bt["gross_exposure"].mean()),
        "avg_net_exposure": float(bt["net_exposure"].mean()),
        "avg_max_weight": float(bt["max_weight"].mean()),
        "n_days": int(len(bt))
    }

## 7. Assess optimization of parameters

Optimization is handled conservatively. The purpose is not to search for the best-looking historical configuration, but to determine whether a small set of economically interpretable parameters changes the conclusion materially.

### 7.1 Parameters assessed

The notebook focuses on the few parameters that matter most economically:
- **logistic regularization (`C`)** for the primary signal model,
- **portfolio breadth (`top_n`)** for the ranking rule,
- and the **Tetlock-rule lookback** where relevant.

### 7.2 Optimization hypothesis and primary tests

This section uses two primary tests.

1. **Validation-slice tuning test.**  
   A narrow grid is evaluated on a validation slice inside each training window. This asks which parameter choices look sensible before the final test period is touched.

2. **Local-stability test.**  
   The preferred conclusion should survive nearby parameter choices. If the apparent edge appears only at one isolated setting, that is evidence against robustness.


In [ ]:
def sorted_dates(panel: pd.DataFrame) -> List[pd.Timestamp]:
    return sorted(pd.to_datetime(panel["calendar_date"]).unique().tolist())

def make_walk_forward_folds(panel: pd.DataFrame, train_min_days: int, test_days: int, step_days: Optional[int] = None):
    dates = sorted_dates(panel)
    if step_days is None:
        step_days = test_days

    folds = []
    start_test = train_min_days
    while start_test + test_days <= len(dates):
        train_dates = dates[:start_test]
        test_dates = dates[start_test:start_test + test_days]
        folds.append((train_dates, test_dates))
        start_test += step_days
    return folds

def split_train_validation(train_df: pd.DataFrame, validation_days: int):
    dates = sorted(pd.to_datetime(train_df["calendar_date"]).unique().tolist())
    if len(dates) <= validation_days + 20:
        return train_df.copy(), pd.DataFrame(columns=train_df.columns)
    cutoff_dates = set(dates[-validation_days:])
    val_df = train_df[train_df["calendar_date"].isin(cutoff_dates)].copy()
    tr_df = train_df[~train_df["calendar_date"].isin(cutoff_dates)].copy()
    return tr_df, val_df

def fit_and_score(train_df, test_df, features, model_factory):
    X_train = train_df[features]
    y_train = train_df["target_up"].astype(int)
    X_test = test_df[features]
    y_test = test_df["target_up"].astype(int)

    model = model_factory()
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        score = model.decision_function(X_test)
        proba = 1 / (1 + np.exp(-score))

    pred = (proba >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan,
    }
    metrics.update(pesaran_timmermann_test(y_test, pred))

    out = test_df[["calendar_date", "ticker", "target_return", "target_up"]].copy()
    out["pred_proba"] = proba
    out["pred_label"] = pred
    return model, metrics, out

def run_walk_forward(panel: pd.DataFrame, feature_set_name: str, model_name: str, model_factory):
    features = FEATURE_SETS[feature_set_name]
    use_cols = ["calendar_date", "ticker", "target_up", "target_return"] + features
    data = panel[use_cols].dropna().copy()
    data["calendar_date"] = pd.to_datetime(data["calendar_date"])

    folds = make_walk_forward_folds(
        data,
        train_min_days=CONFIG["train_min_days"],
        test_days=CONFIG["test_days"],
        step_days=CONFIG["test_days"],
    )
    if not folds:
        raise ValueError("No walk-forward folds created. Reduce train/test requirements or expand the sample.")

    fold_metrics = []
    fold_preds = []

    for fold_idx, (train_dates, test_dates) in enumerate(folds, start=1):
        train_df = data[data["calendar_date"].isin(train_dates)].copy()
        test_df = data[data["calendar_date"].isin(test_dates)].copy()

        train_core, val_df = split_train_validation(train_df, CONFIG["validation_days"])
        _, metrics, pred_df = fit_and_score(train_core, test_df, features, model_factory)

        metrics["fold"] = fold_idx
        metrics["n_train"] = len(train_core)
        metrics["n_test"] = len(test_df)
        metrics["start_test"] = min(test_dates)
        metrics["end_test"] = max(test_dates)
        fold_metrics.append(metrics)

        pred_df["fold"] = fold_idx
        pred_df["feature_set"] = feature_set_name
        pred_df["model_type"] = model_name
        fold_preds.append(pred_df)

    metrics_df = pd.DataFrame(fold_metrics)
    preds_df = pd.concat(fold_preds, axis=0, ignore_index=True)
    return metrics_df, preds_df

In [ ]:
PRIMARY_TUNING_CONFIG = {
    "sentiment_source": "finbert" if "finbert" in model_panels else list(model_panels.keys())[0],
    "feature_set": "sentiment_plus_controls",
    "cost_bps": 5,
    "logistic_C_grid": [0.1, 1.0, 10.0],
    "top_n_grid": [3, 5, 10],
}

def make_logistic_model_with_C(C: float):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, C=C, class_weight="balanced"))
    ])

def run_primary_parameter_sweep(panel: pd.DataFrame, tuning_config: Dict) -> pd.DataFrame:
    features = FEATURE_SETS[tuning_config["feature_set"]]
    use_cols = ["calendar_date", "ticker", "target_up", "target_return"] + features
    data = panel[use_cols].dropna().copy()
    data["calendar_date"] = pd.to_datetime(data["calendar_date"])

    folds = make_walk_forward_folds(
        data,
        train_min_days=CONFIG["train_min_days"],
        test_days=CONFIG["test_days"],
        step_days=CONFIG["test_days"],
    )

    rows = []
    for fold_idx, (train_dates, _) in enumerate(folds, start=1):
        train_df = data[data["calendar_date"].isin(train_dates)].copy()
        train_core, val_df = split_train_validation(train_df, CONFIG["validation_days"])
        if val_df.empty:
            continue

        for C in tuning_config["logistic_C_grid"]:
            _, metrics, val_pred = fit_and_score(
                train_core,
                val_df,
                features,
                lambda C=C: make_logistic_model_with_C(C)
            )
            for top_n in tuning_config["top_n_grid"]:
                bt = cross_sectional_long_short(val_pred, cost_bps=tuning_config["cost_bps"], top_n=top_n)
                summary = summarize_backtest(bt, label=f"val_fold_{fold_idx}")
                rows.append({
                    "fold": fold_idx,
                    "C": C,
                    "top_n": top_n,
                    "val_accuracy": metrics.get("accuracy", np.nan),
                    "val_balanced_accuracy": metrics.get("balanced_accuracy", np.nan),
                    "val_auc": metrics.get("auc", np.nan),
                    "val_hit_rate": metrics.get("hit_rate", np.nan),
                    "val_pt_stat": metrics.get("pt_stat", np.nan),
                    "val_p_value": metrics.get("p_value", np.nan),
                    "val_sharpe": summary.get("sharpe", np.nan),
                    "val_annual_return": summary.get("annual_return", np.nan),
                    "val_avg_turnover": summary.get("avg_turnover", np.nan),
                })
    return pd.DataFrame(rows)

parameter_sweep_df = pd.DataFrame()
if model_panels:
    primary_panel_for_tuning = model_panels[PRIMARY_TUNING_CONFIG["sentiment_source"]]
    parameter_sweep_df = run_primary_parameter_sweep(primary_panel_for_tuning, PRIMARY_TUNING_CONFIG)
    save_csv(parameter_sweep_df, "parameter_sweep_validation_results.csv")

if not parameter_sweep_df.empty:
    display(parameter_sweep_df.head())
    selected_by_fold = (
        parameter_sweep_df.sort_values(["fold", "val_sharpe", "val_balanced_accuracy"], ascending=[True, False, False])
        .groupby("fold")
        .head(1)
        .reset_index(drop=True)
    )
    save_csv(selected_by_fold, "parameter_sweep_selected_by_fold.csv")
    display(selected_by_fold)

    selection_stability = (
        selected_by_fold.groupby(["C", "top_n"]).size().reset_index(name="n_folds_selected").sort_values("n_folds_selected", ascending=False)
    )
    save_csv(selection_stability, "parameter_selection_stability.csv")
    display(selection_stability)
else:
    print("Parameter sweep skipped because the model panel is empty.")

### 7.3 Parameter optimization test analysis

The parameter-assessment results suggest that optimization should be interpreted as a robustness screen rather than a search for a single best setting. In the validation-slice tuning test, there are pockets of strong-looking performance—especially for narrower portfolios such as top_n = 3, which occasionally produce the highest validation Sharpe and annual return—but these gains are not consistently aligned with the best predictive metrics, and several folds show weak or even negative PT statistics and returns. That divergence indicates that the most profitable-looking configuration in one window is not always the most statistically convincing, which is exactly why aggressive optimization would be misleading. The local-stability table is more informative: selection is concentrated in a small neighborhood rather than a single knife-edge optimum, with (C = 10, top_n = 3) and (C = 1, top_n = 10) each selected in 7 folds, and nearby settings such as (C = 0.1, top_n = 3) and (C = 0.1, top_n = 10) also recurring multiple times. That pattern implies the strategy is somewhat sensitive to breadth and regularization, but not in a way that points to one fragile, isolated parameter pair. The economically relevant insight is that extremely concentrated portfolios can improve headline returns in some folds at the cost of higher turnover and greater instability, while broader portfolios tend to deliver a flatter but more stable profile. Overall, the optimization evidence supports a shallow and noisy optimum, not a sharply identified best configuration; accordingly, the preferred parameter choice should be taken from the recurring, theory-consistent region of the grid rather than from the single highest historical validation score.

## 8. Walk-forward analysis, objective function, and OOS evaluation

Walk-forward evaluation is the central empirical process of the study - the strategy is repeatedly trained on past data and judged on future data that were not available at estimation time. This is the main guardrail against strategy claims built on one helpful historical segment.


### 8.1 Objective function

The notebook separates objectives by layer:

- **signal layer:** discrimination and probability quality,
- **rule layer:** net economic value after costs,
- **global interpretation:** consistency across folds rather than one headline number.

This distinction matters because a parameter set that maximizes classification quality can still monetize poorly if it induces high turnover or concentration. Conversely, a slightly weaker predictive model can sometimes yield a better trading rule because it is smoother and cheaper to implement.

### 8.2 Walk-forward hypothesis and primary tests

This section uses two primary tests.

1. **Walk-forward predictive test.**  
   Evaluate the forecast repeatedly out of sample across rolling-origin folds.

2. **Objective-alignment test.**  
   Compare whether the specification favored by predictive metrics remains competitive when judged by net trading metrics.

In [ ]:
all_predictive_results = []
all_prediction_panels = []

model_factories = {
    "logistic": make_logistic_model,
    "xgboost": make_xgb_model,
}

for sentiment_source, panel in model_panels.items():
    for feature_set_name in FEATURE_SETS.keys():
        for model_type, model_factory in model_factories.items():
            print(f"Running: source={sentiment_source} | features={feature_set_name} | model={model_type}")
            try:
                metrics_df, preds_df = run_walk_forward(panel, feature_set_name, model_type, model_factory)
                metrics_df["sentiment_source"] = sentiment_source
                metrics_df["feature_set"] = feature_set_name
                metrics_df["model_type"] = model_type
                all_predictive_results.append(metrics_df)

                preds_df["sentiment_source"] = sentiment_source
                all_prediction_panels.append(preds_df)
            except Exception as e:
                print(f"Skipped combination due to error: {e}")

predictive_results = pd.concat(all_predictive_results, axis=0, ignore_index=True) if all_predictive_results else pd.DataFrame()
prediction_panel = pd.concat(all_prediction_panels, axis=0, ignore_index=True) if all_prediction_panels else pd.DataFrame()

save_csv(predictive_results, "predictive_results_by_fold.csv")
save_csv(prediction_panel, "prediction_panel.csv")

predictive_results.head()

In [ ]:
if not predictive_results.empty:
    summary_predictive = (
        predictive_results.groupby(["sentiment_source", "feature_set", "model_type"])
        [["auc", "balanced_accuracy", "pt_stat", "p_value", "hit_rate"]]
        .mean()
        .sort_values(["auc", "balanced_accuracy"], ascending=False)
        .reset_index()
    )
    save_csv(summary_predictive, "predictive_results_summary.csv")
    display(summary_predictive)

### 8.3 Convert walk-forward predictions into portfolio returns

These cells apply the previously defined rules to the out-of-sample prediction stream and produce implementable strategy returns, turnover, and exposure statistics.

In [ ]:
backtest_summaries = []
backtest_paths = []

if not prediction_panel.empty:
    grouped = prediction_panel.groupby(["sentiment_source", "feature_set", "model_type"])
    for (sentiment_source, feature_set, model_type), pred_df in grouped:
        for cost_bps in CONFIG["cost_bps_list"]:
            bt = cross_sectional_long_short(pred_df, cost_bps=cost_bps, top_n=CONFIG["top_n"])
            label = f"{sentiment_source} | {feature_set} | {model_type} | {cost_bps}bps"
            summary = summarize_backtest(bt, label)
            summary["sentiment_source"] = sentiment_source
            summary["feature_set"] = feature_set
            summary["model_type"] = model_type
            summary["cost_bps"] = cost_bps
            backtest_summaries.append(summary)

            bt["strategy_label"] = label
            bt["sentiment_source"] = sentiment_source
            bt["feature_set"] = feature_set
            bt["model_type"] = model_type
            bt["cost_bps"] = cost_bps
            backtest_paths.append(bt)

backtest_summary_df = pd.DataFrame(backtest_summaries)
backtest_paths_df = pd.concat(backtest_paths, axis=0, ignore_index=True) if backtest_paths else pd.DataFrame()

save_csv(backtest_summary_df, "ml_backtest_summary.csv")
save_csv(backtest_paths_df, "ml_backtest_paths.csv")

backtest_summary_df.sort_values("sharpe", ascending=False).head(20)

In [ ]:
if not backtest_paths_df.empty:
    best_row = backtest_summary_df.sort_values("sharpe", ascending=False).iloc[0]
    mask = (
        (backtest_paths_df["sentiment_source"] == best_row["sentiment_source"]) &
        (backtest_paths_df["feature_set"] == best_row["feature_set"]) &
        (backtest_paths_df["model_type"] == best_row["model_type"]) &
        (backtest_paths_df["cost_bps"] == best_row["cost_bps"])
    )
    best_bt = backtest_paths_df[mask].copy()
    equity = (1 + best_bt["net_return"].fillna(0)).cumprod()
    plt.figure(figsize=(12, 5))
    plt.plot(best_bt["calendar_date"], equity)
    plt.title(f"Best ML strategy equity curve\n{best_row['strategy']}")
    plt.ylabel("Growth of $1")
    plt.show()

### 8.4 Benchmark rule: Tetlock-style tercile comparator

A simple comparator is essential. The tercile rule below uses only rolling pessimism information and therefore represents the most direct translation of the original media-pessimism idea into a trade. It does not rely on estimated probabilities, flexible ML, or many control variables.

This benchmark serves two purposes. First, it shows whether the economic mechanism has any value in a low-complexity form. Second, it helps interpret the more sophisticated ML strategy: if the model-based rule outperforms, the gain can be attributed to better signal extraction or portfolio ranking rather than merely to the existence of a sentiment effect itself.



In [ ]:
tercile_summaries = []
tercile_paths = []

for sentiment_source, panel in model_panels.items():
    sig_df = rolling_tercile_signal(panel, lookback=252)
    for cost_bps in CONFIG["cost_bps_list"]:
        bt = equal_weight_signal_backtest(sig_df, cost_bps=cost_bps)
        label = f"{sentiment_source} | tercile_rule | {cost_bps}bps"
        summary = summarize_backtest(bt, label)
        summary["sentiment_source"] = sentiment_source
        summary["feature_set"] = "tercile_rule"
        summary["model_type"] = "rule_based"
        summary["cost_bps"] = cost_bps
        tercile_summaries.append(summary)

        bt["strategy_label"] = label
        bt["sentiment_source"] = sentiment_source
        bt["feature_set"] = "tercile_rule"
        bt["model_type"] = "rule_based"
        bt["cost_bps"] = cost_bps
        tercile_paths.append(bt)

tercile_summary_df = pd.DataFrame(tercile_summaries)
tercile_paths_df = pd.concat(tercile_paths, axis=0, ignore_index=True) if tercile_paths else pd.DataFrame()

save_csv(tercile_summary_df, "tercile_backtest_summary.csv")
save_csv(tercile_paths_df, "tercile_backtest_paths.csv")

tercile_summary_df.sort_values("sharpe", ascending=False).head(20)

### 6.3: Rule process test analysis

The rule-process results provide the clearest evidence that the strategy’s value lies in the forecast-to-ranking translation rather than in the raw indicator alone. The primary cross-sectional ranking rule performs meaningfully better than the simpler Tetlock-style rolling-tercile benchmark across essentially every relevant dimension. At zero transaction costs, the best ranking specification—lex | sentiment_only | logistic—delivers an annual return of about 14.0%, annual volatility of 16.2%, and a Sharpe ratio of 0.89, with a tolerable maximum drawdown of roughly -24%. By contrast, the benchmark tercile rules are weak even before costs: the best of them, lex | tercile_rule | 0bps, earns only about 1.8% annually with a Sharpe near 0.20, while the FinBERT and SieBERT tercile variants are already negative gross. This strongly supports the benchmark-rule comparison hypothesis: the forecasting layer adds substantial economic value relative to a direct rule on raw sentiment. At the same time, the implementation profile is not trivial. The ranking portfolios exhibit high turnover—roughly 3.0 on average per day for the leading variants—so the gross results depend heavily on friction assumptions. That means the rule process is successful in extracting a tradable ordering from the signal, but it does so through an active and potentially costly implementation. The overall interpretation is therefore not that raw news sentiment is directly monetizable, but that it becomes economically useful when transformed into a continuous cross-sectional forecast and embedded in a structured ranking portfolio.

### 8.5: Walk forward analysis

The walk-forward results reinforce that conclusion, but also impose an important qualification about objective alignment. On the predictive side, the best out-of-sample discrimination comes from the sentiment-only logistic models, especially SieBERT and lex, with AUCs around 0.508 and balanced accuracy just above 0.505, while baseline market-only logistic models are consistently worse and even slightly sub-random on average. This implies that sentiment contributes a small but repeatable forecasting edge out of sample. The more important question is whether the specifications that forecast best also monetize best. The answer is only partial: the economically strongest rule is the lex sentiment-only logistic strategy, which is broadly consistent with the better predictive models, but the edge is extremely sensitive to transaction costs. At 0 bps the strategy looks attractive, yet at 5 bps its annual return turns negative and the Sharpe falls below zero, indicating that the gross signal is not strong enough to survive even moderate frictions under the current turnover profile. That makes the final walk-forward verdict a qualified rather than clean success. The repeated rolling-origin tests do show a real, out-of-sample signal and a rule that can exploit it in gross terms, but the objective-alignment test reveals that statistical usefulness does not robustly translate into net profitability. Overall, the strategy remains credible as a forecasting and ranking exercise, but its economic viability depends critically on implementation improvements, lower trading costs, or a smoother rule that preserves signal quality while reducing turnover.

## 9. Assess overfitting

### 9.1 Overfitting hypothesis and primary tests

This section is comprised of two primary tests.

1. **Time-stability test.**  
   Decompose the preferred strategy by fold and/or calendar year to determine whether performance is broad-based or concentrated in a small subset of periods.

2. **Fragility test.**  
   Examine whether the result disappears under modest cost changes or nearby parameter choices.

In [ ]:
if not backtest_paths_df.empty:
    tmp = backtest_paths_df.copy()
    tmp["year"] = pd.to_datetime(tmp["calendar_date"]).dt.year
    yearly = (
        tmp.groupby(["strategy_label", "year"])["net_return"]
        .apply(lambda s: (1 + s.fillna(0)).prod() - 1)
        .reset_index()
    )
    save_csv(yearly, "yearly_strategy_returns.csv")
    display(yearly.head())

In [ ]:
overfitting_tables = {}

if not predictive_results.empty:
    fold_dispersion = (
        predictive_results.groupby(["sentiment_source", "feature_set", "model_type"])
        .agg(
            mean_auc=("auc", "mean"),
            std_auc=("auc", "std"),
            mean_bal_acc=("balanced_accuracy", "mean"),
            std_bal_acc=("balanced_accuracy", "std"),
            mean_hit_rate=("hit_rate", "mean"),
            std_hit_rate=("hit_rate", "std"),
            n_folds=("fold", "nunique")
        )
        .reset_index()
        .sort_values(["mean_auc", "mean_bal_acc"], ascending=False)
    )
    overfitting_tables["fold_dispersion"] = fold_dispersion
    save_csv(fold_dispersion, "overfitting_fold_dispersion.csv")
    display(fold_dispersion.head(15))

if not backtest_summary_df.empty:
    cost_fragility = (
        backtest_summary_df.groupby(["sentiment_source", "feature_set", "model_type"])
        .apply(lambda g: pd.Series({
            "sharpe_at_0bps": g.loc[g["cost_bps"].eq(0), "sharpe"].max() if (g["cost_bps"] == 0).any() else np.nan,
            "sharpe_at_25bps": g.loc[g["cost_bps"].eq(25), "sharpe"].max() if (g["cost_bps"] == 25).any() else np.nan,
            "annual_return_at_0bps": g.loc[g["cost_bps"].eq(0), "annual_return"].max() if (g["cost_bps"] == 0).any() else np.nan,
            "annual_return_at_25bps": g.loc[g["cost_bps"].eq(25), "annual_return"].max() if (g["cost_bps"] == 25).any() else np.nan,
        }))
        .reset_index()
    )
    cost_fragility["sharpe_decay_0_to_25bps"] = cost_fragility["sharpe_at_0bps"] - cost_fragility["sharpe_at_25bps"]
    overfitting_tables["cost_fragility"] = cost_fragility.sort_values("sharpe_decay_0_to_25bps")
    save_csv(cost_fragility, "overfitting_cost_fragility.csv")
    display(cost_fragility.sort_values("sharpe_at_25bps", ascending=False).head(15))

if not parameter_sweep_df.empty:
    parameter_instability = (
        parameter_sweep_df.groupby("fold")[["val_sharpe", "val_balanced_accuracy", "val_auc"]].max().reset_index()
    )
    overfitting_tables["parameter_instability"] = parameter_instability
    save_csv(parameter_instability, "overfitting_parameter_instability.csv")
    display(parameter_instability)


### 9.2 Overfitting test analysis

The overfitting diagnostics indicate the strategy is statistically present but economically fragile, so the right conclusion is caution over rejection. On the time-stability side, the preferred signal specifications show relatively stable but modest predictive performance across folds—for example, the best sentiment-only logistic models have mean AUCs just above 0.507–0.508 with fold-level standard deviations around 0.019–0.021, which suggests the forecasting edge is not concentrated in a single isolated split. At the same time, the parameter-instability table shows that validation Sharpe varies dramatically across folds, ranging from strongly positive to sharply negative, even while balanced accuracy and AUC remain clustered in a narrow band near 0.50. The divergence indicates that the economic outcome is much less stable than the predictive signal. The yearly return decomposition points to a similar result: performance is not driven by only one spectacular year, but it is clearly uneven over time, with meaningful variation between favorable and unfavorable regimes. The fragility test is more decisive. Cost sensitivity is severe across essentially all variants: strategies with respectable gross Sharpe at 0 bps collapse to strongly negative Sharpe ratios by 25 bps, and the best-performing lex sentiment-only logistic rule falls from a Sharpe near 0.89 to roughly -4.84. Nearby parameter choices also do not preserve economic performance smoothly enough to fully dismiss data-mining concerns. Overall, these results suggest that the underlying predictive relationship is not obviously overfit in the narrow statistical sense, since it persists weakly across folds, but the monetized strategy is overfit to favorable implementation assumptions because its apparent edge deteriorates abruptly once realistic frictions and parameter sensitivity are introduced.

## 10. Extensions

This section includes 2 extensions that deepen interpretation and call back to some of Tetlock's original ideas.

### 10.1 Extension logic

The extensions are chosen to answer two questions:

1. **Mechanism extension:** does the return effect look like short-run pressure and subsequent reversal?
2. **Model extension:** do the conclusions depend on using a finance-specific sentiment source rather than a generic or lexicon benchmark?

### 10.2 Extension: reversal across horizons \(t+1\) to \(t+5\)

The next-day sign forecast is only one manifestation of the sentiment mechanism. Tetlock-style theories often imply that unusually negative language can create short-run pressure followed by partial reversal as attention and liquidity effects dissipate. The horizon extension therefore tests whether the relation between pessimism and returns changes as the holding period is lengthened from one to five days.

This extension is important because it distinguishes a purely statistical one-day pattern from a more coherent economic story about temporary sentiment pressure.



In [ ]:
def add_forward_horizons(panel: pd.DataFrame, max_horizon: int = 5) -> pd.DataFrame:
    df = panel.sort_values(["ticker", "calendar_date"]).copy()
    for h in range(1, max_horizon + 1):
        vals = []
        for _, g in df.groupby("ticker"):
            r = g["target_return"].copy().reset_index(drop=True)
            cum = pd.Series(index=r.index, dtype=float)
            for i in range(len(r)):
                if i + h - 1 < len(r):
                    cum.iloc[i] = r.iloc[i:i+h].sum()
            vals.append(cum.values)
        df[f"fwd_ret_{h}d"] = np.concatenate(vals)
    return df

def pooled_horizon_regression(panel: pd.DataFrame, max_horizon: int = 5) -> pd.DataFrame:
    df = add_forward_horizons(panel, max_horizon=max_horizon)
    rows = []
    for h in range(1, max_horizon + 1):
        cols = ["pessimism_mean", f"fwd_ret_{h}d", "ret_lag_1", "ret_lag_2", "ret_lag_3", "log_vol_lag_1", "january_dummy"]
        tmp = df[cols].dropna().copy()
        if len(tmp) < 100:
            continue
        X = sm.add_constant(tmp[["pessimism_mean", "ret_lag_1", "ret_lag_2", "ret_lag_3", "log_vol_lag_1", "january_dummy"]])
        y = tmp[f"fwd_ret_{h}d"]
        fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
        rows.append({
            "horizon_days": h,
            "coef_pessimism": fit.params.get("pessimism_mean", np.nan),
            "tstat_pessimism": fit.tvalues.get("pessimism_mean", np.nan),
            "pvalue_pessimism": fit.pvalues.get("pessimism_mean", np.nan),
            "n_obs": int(fit.nobs)
        })
    return pd.DataFrame(rows)

horizon_results = []
for sentiment_source, panel in model_panels.items():
    try:
        tmp = pooled_horizon_regression(panel, max_horizon=5)
        tmp["sentiment_source"] = sentiment_source
        horizon_results.append(tmp)
    except Exception as e:
        print(f"Horizon test failed for {sentiment_source}: {e}")

horizon_results_df = pd.concat(horizon_results, axis=0, ignore_index=True) if horizon_results else pd.DataFrame()
save_csv(horizon_results_df, "horizon_reversal_results.csv")
horizon_results_df

In [ ]:
if not horizon_results_df.empty:
    for source, g in horizon_results_df.groupby("sentiment_source"):
        plt.plot(g["horizon_days"], g["coef_pessimism"], marker="o", label=source)
    plt.axhline(0, color="black", linewidth=1)
    plt.title("Pessimism coefficient across return horizons")
    plt.xlabel("Forward horizon (days)")
    plt.ylabel("Coefficient on pessimism")
    plt.legend()
    plt.show()

### 10.3 Extension 1 Analysis

The horizon-extension results do not provide meaningful evidence for a Tetlock-style short-run pressure followed by reversal mechanism. Across all three sentiment sources, the pessimism coefficients are small, statistically insignificant at every horizon from t+1 to t+5, and do not trace out a clean sign pattern that would support an initial negative impact followed by systematic rebound. FinBERT remains mildly positive and fairly stable across horizons, SieBERT stays essentially at zero throughout, and the lexicon benchmark fluctuates more sharply but in an erratic way with large noise and no statistical support. In other words, once the same protocol is applied across forward horizons, the data do not show a coherent temporary-pressure profile; instead, they suggest that any horizon dependence is weak relative to noise. This extension therefore refines the main conclusion: the project’s signal appears better understood as a modest cross-sectional forecasting device than as strong evidence for any structural reversal, and the finance-specific models do not gain a decisive advantage on this mechanism test.

### 10.4 Extension: activity / volume channel

A second extension asks whether stronger sentiment—especially stronger *absolute* tone—is associated with higher subsequent trading activity. This is a mechanism test rather than a performance test. If sentiment primarily works by attracting investor attention or increasing disagreement, then its effect may show up more clearly in volume than in unconditional next-day return.

A positive and robust activity relation strengthens the interpretation of the strategy even if return effects are modest.



In [ ]:
def volume_channel_regression(panel: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "abs_sentiment_mean", "next_log_volume", "log_vol_lag_1",
        "realized_abs_ret_lag_1", "january_dummy"
    ]
    tmp = panel[cols].dropna().copy()
    if len(tmp) < 100:
        return pd.DataFrame()

    X = sm.add_constant(tmp[["abs_sentiment_mean", "log_vol_lag_1", "realized_abs_ret_lag_1", "january_dummy"]])
    y = tmp["next_log_volume"]
    fit = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": 5})

    return pd.DataFrame([{
        "coef_abs_sentiment": fit.params.get("abs_sentiment_mean", np.nan),
        "tstat_abs_sentiment": fit.tvalues.get("abs_sentiment_mean", np.nan),
        "pvalue_abs_sentiment": fit.pvalues.get("abs_sentiment_mean", np.nan),
        "r2": fit.rsquared,
        "n_obs": int(fit.nobs)
    }])

volume_results = []
for source, panel in model_panels.items():
    try:
        vr = volume_channel_regression(panel)
        if not vr.empty:
            vr["sentiment_source"] = source
            volume_results.append(vr)
    except Exception as e:
        print(f"Volume channel failed for {source}: {e}")

volume_results_df = pd.concat(volume_results, axis=0, ignore_index=True) if volume_results else pd.DataFrame()
save_csv(volume_results_df, "volume_channel_results.csv")
volume_results_df

### 10.5 Extension 2 Analysis

The activity-channel extension provides some of the strongest mechanism evidence in the study and is one of the clearest places where the finance-specific sentiment measure appears to matter. For FinBERT, absolute sentiment is positively and highly significantly related to next-day log volume ($\beta \approx 0.044$, $t \approx 5.78$, $p < 10^{-8}$), which is exactly the pattern one would expect if emotionally intense financial news increases investor attention, disagreement, or trading urgency. By contrast, the same specification produces coefficients near zero and statistically insignificant for SieBERT and the lexicon benchmark, suggesting that not all sentiment measures capture this mechanism equally well. This result therefore supports the model-extension hypothesis: once the same protocol is applied across alternatives, the finance-specific model retains a meaningful advantage in detecting an economically plausible attention channel. It strengthens the interpretation that the signal may work indirectly through market activity and information processing, with FinBERT doing a better job than the generic or lexicon-based alternatives at identifying news that actually moves trading behavior.

## Summary tables

In [ ]:
tables = {}

if not predictive_results.empty:
    tables["predictive"] = (
        predictive_results.groupby(["sentiment_source", "feature_set", "model_type"])
        [["accuracy", "balanced_accuracy", "auc", "hit_rate", "pt_stat", "p_value"]]
        .mean()
        .sort_values(["auc", "balanced_accuracy"], ascending=False)
        .reset_index()
    )
    display(tables["predictive"])

if not backtest_summary_df.empty:
    tables["ml_backtest"] = backtest_summary_df.sort_values("sharpe", ascending=False).reset_index(drop=True)
    display(tables["ml_backtest"].head(20))

if not tercile_summary_df.empty:
    tables["tercile_backtest"] = tercile_summary_df.sort_values("sharpe", ascending=False).reset_index(drop=True)
    display(tables["tercile_backtest"].head(20))

if not horizon_results_df.empty:
    tables["horizon"] = horizon_results_df.copy()
    display(tables["horizon"])

if not volume_results_df.empty:
    tables["volume"] = volume_results_df.copy()
    display(tables["volume"])

## Conclusion

Across the full study, the central empirical question was whether modern news-based sentiment measures can improve short-horizon equity forecasting and whether that predictive content survives translation into a realistic trading rule. The primary specification that emerged most favorably was a sentiment-only cross-sectional ranking strategy, especially the logistic models built on lexicon and SieBERT sentiment, with FinBERT remaining competitive in some mechanism tests. The main finding is that sentiment contains a small but detectable forecasting signal, and that signal can be monetized in gross terms through a ranking portfolio, but the economic edge is fragile once realistic trading frictions are imposed. At the indicator level, raw pessimism alone had very limited unconditional return-predictive power: pooled return correlations were near zero and the horizon-extension tests showed no clean short-run pressure/reversal pattern from t+1 to t+5. However, the activity-channel evidence was more supportive. In particular, FinBERT showed a strong and highly significant positive link between absolute sentiment and next-day volume, suggesting that finance-specific language is especially useful for capturing an attention or trading-activity mechanism even when it is less effective at direct return prediction. Taken together, the indicator layer was weak but economically plausible, with modest informational content that required further modeling to become useful.

<br>

At the signal-process level, the best out-of-sample forecasting results came from the sentiment-only logistic models, with SieBERT posting the highest AUC (about 0.5081) and lexicon very close behind (AUC about 0.5077, with the highest balanced accuracy and hit rate). FinBERT lagged on pure forecast metrics, while the richer sentiment-plus-controls specifications generally did not improve results and often underperformed the simpler sentiment-only alternatives. This pattern suggests that the useful information in the panel was better extracted by simpler linear ranking models than by adding more controls or nonlinear flexibility. The ranking-value evidence was more encouraging than the classification statistics alone: the best signal specifications generated positive top-minus-bottom spreads and, when translated into the primary long-short ranking rule, produced economically meaningful gross performance. The strongest result came from lex | sentiment_only | logistic, with roughly 14.0% annual return, 16.2% annual volatility, and a 0.89 Sharpe at zero costs, clearly outperforming the low-complexity Tetlock-style tercile comparator, whose best gross Sharpe was only about 0.20 and was negative for most alternatives. This implies that the forecasting layer added real value beyond a direct rule on raw pessimism. In relative terms, the models split into two roles: lexicon and SieBERT performed best as forecasting/ranking tools, while FinBERT performed best on the volume-based mechanism test, giving the finance-specific model more interpretive strength than pure trading superiority.

<br>

The final economic conclusion must be qualified by the optimization, walk-forward, and overfitting evidence. Parameter assessment suggested a shallow and noisy optimum rather than a single knife-edge best setting, which is a positive sign, and fold-level AUCs for the stronger signal models were fairly stable, indicating that the underlying predictive edge was not purely a one-period accident. But the monetized rule proved fragile to implementation assumptions: turnover for the ranking strategies was high (around 3.0 on average), and the attractive gross Sharpe ratios at 0 bps collapsed once even modest transaction costs were introduced. For example, the best lex sentiment-only logistic strategy turned negative by 5 bps, while cost-fragility tests showed large Sharpe decay across essentially all models. The extensions reinforced this mixed conclusion. The reversal-horizon exercise did not support a clean Tetlock-style temporary-pressure-and-reversal story, but the volume-channel test did show that FinBERT uniquely captured a strong attention/activity effect. Overall, the study supports a refined conclusion: news sentiment is statistically and economically informative enough to improve cross-sectional ranking forecasts, but not strong enough in this implementation to deliver robust net trading profits after realistic frictions. Among the indicators, lexicon and SieBERT were strongest for gross predictive and rule performance, while FinBERT was strongest for identifying the underlying activity mechanism, so the choice of sentiment model matters depending on whether the objective is forecasting accuracy, tradability, or economic interpretation. Future directions may include expanding the transformer inference specs (which were limited for computational scope) to see if FinBERT/SieBERT can capture more nuance, utilizing full article bodies instead of just headlines for sentiment complexity, and using more robust pricing data (OHLCV) in tandem with sentiment for the combined strategies.